<a href="https://colab.research.google.com/github/jhgfdsa1256/kivy-test/blob/main/mafia42_replay_clawler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install playwright beautifulsoup4 -q
!playwright install chromium --with-deps

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:7 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libasound2t64 is already the newest version (1.2.11-1ubuntu0.3).
libatk-bridge2.0-0t64 is already the newest version (2.52.0-1build1).
libatk1.

In [10]:
REPLAY_URL = "https://mafia42.com/history/kr/191bd6f52ff0d67516f03a5651aa918a"

In [11]:
import re
import json
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright


async def get_rendered_replay_html(history_url: str) -> str:
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        api_url_holder = {}

        def handle_request(req):
            if "GetMafiaChat" in req.url or "lambda-url" in req.url:
                api_url_holder["url"] = req.url

        page.on("request", handle_request)

        await page.goto(history_url, wait_until="networkidle", timeout=30000)
        await page.wait_for_timeout(2000)

        api_url = api_url_holder.get("url")
        if not api_url:
            await browser.close()
            raise RuntimeError("GetMafiaChat API 주소를 찾지 못했습니다. URL이 유효한지 확인하세요.")

        await page.goto(api_url, wait_until="networkidle", timeout=30000)
        await page.wait_for_timeout(3000)

        rendered_html = await page.content()
        await browser.close()

    return rendered_html


def parse_replay_html(html: str):
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("section", class_="table")
    events = []

    if table is None:
        return events

    for child in table.find_all(recursive=False):
        classes = child.get("class", []) or []

        if "system" in classes:
            events.append({
                "event_type": "system",
                "text": child.get_text(strip=True)
            })

        elif "chat-data-container" in classes:
            job_img = child.find("img", class_="job-icon-img")
            job_name = None
            if job_img and job_img.get("src"):
                m = re.search(r"jobthumb_([a-zA-Z]+)\.png", job_img["src"])
                if m:
                    job_name = m.group(1)

            nickname_el = child.find("div", class_="nick-name")
            nickname = nickname_el.get_text(strip=True) if nickname_el else None

            chat_container = child.find("div", class_=lambda c: c and "chat-container" in c)
            if chat_container:
                bubbles = chat_container.find_all(
                    "div", class_=lambda c: c and "chat-bubble" in c
                )
                for bubble in bubbles:
                    b_classes = bubble.get("class", [])
                    chat_type = "CHAT"
                    for t in ["MAFIACHAT", "GHOSTCHAT", "MEGAPHONE", "CHAT"]:
                        if t in b_classes:
                            chat_type = t
                            break
                    events.append({
                        "event_type": "chat",
                        "chat_type": chat_type,
                        "nickname": nickname,
                        "job": job_name,
                        "message": bubble.get_text(strip=True)
                    })

        elif "secret-letter-container" in classes:
            to_el = child.find("div", class_="to")
            from_el = child.find("div", class_="from")
            msg_el = child.find("div", class_="message")
            events.append({
                "event_type": "secret_letter",
                "to": to_el.get_text(strip=True).replace("To.", "") if to_el else None,
                "from": from_el.get_text(strip=True).replace("from.", "") if from_el else None,
                "message": msg_el.get_text(strip=True) if msg_el else None
            })

    return events


def parse_player_list(html: str):
    soup = BeautifulSoup(html, "html.parser")
    players = []
    for item in soup.select("#user-table .item"):
        job_img = item.find("img", class_="job-icon-img")
        job_name = None
        if job_img and job_img.get("src"):
            m = re.search(r"jobthumb_([a-zA-Z]+)\.png", job_img["src"])
            if m:
                job_name = m.group(1)
        nickname_el = item.find("div", class_="nick-name")
        players.append({
            "nickname": nickname_el.get_text(strip=True) if nickname_el else None,
            "job": job_name
        })
    return players

In [8]:
rendered_html = await get_rendered_replay_html(REPLAY_URL)

players = parse_player_list(rendered_html)
events = parse_replay_html(rendered_html)

print(f"참가자 수: {len(players)}")
print(f"총 이벤트 수: {len(events)}")

result = {"source_url": REPLAY_URL, "players": players, "events": events}

with open("parsed_replay.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("저장 완료: parsed_replay.json")

참가자 수: 8
총 이벤트 수: 272
저장 완료: parsed_replay.json
